# Ewald construction for diffraction gratings

**How does the change from the incident wavevector to an outgoing order become a geometric construction?**

**Medium and wavelength convention.** The surrounding medium is homogeneous, isotropic and lossless.

- $\lambda$ is wavelength **in the medium**: $\lambda=\lambda_0/n$ and $k=nk_0=2\pi/\lambda$.
- The controls vary medium wavelength; there is no separate index control.

See the [course medium convention](../0_0_Foundations_of_Scalar_Optics.ipynb#medium-and-wavelength).

This standalone notebook holds the material moved from section 7 of [Notebook 5](../5_0_Grating_Diffraction_in_Three_Dimensions.ipynb), together with its coordinate comparison and checkpoint questions. Its place in the series is left open.

Run from top to bottom in this project. The notebook has its own imports, parameters and explorer; no cells or variables from another notebook are required. It uses NumPy, Plotly and ipywidgets (and anywidget for Plotly versions that require it). The Ewald view and the later Harvey comparison share this notebook's controls. Rotations are preserved as parameters change.

We first recall the outgoing orders, then define the scattering vector, construct the sphere and grating rods, and connect those coordinates to the direction-cosine diagram. The editable **small calculation** is independent of the sliders. These constructions determine possible directions, not diffraction efficiencies.


## 1. Recall the waves represented by the construction

The grating lies at $z=0$, repeats along $x$ with period $d$, and has uniform grooves along $y$. Incidence is from $z>0$ and transmission is into $z<0$. Both exterior regions have the same homogeneous, isotropic, lossless medium. Lengths are in micrometres; $\lambda$ is the wavelength in that medium, $k=2\pi/\lambda$, and the time convention is $e^{-i\omega t}$.

For incident component fractions $a_i=k_{x,i}/k$ and $b_i=k_{y,i}/k$,
$$\frac{\mathbf k_i}{k}=(a_i,b_i,-\sqrt{1-a_i^2-b_i^2}),\qquad a_i^2+b_i^2<1.$$

Across the grating, periodicity permits an integer change $mG$, with $G=2\pi/d$; uniformity along the grooves preserves $k_y$:
$$k_{x,m}=k_{x,i}+mG,\qquad k_{y,m}=k_{y,i},\qquad D_m=k^2-k_{x,m}^2-k_{y,m}^2.$$
The outgoing or decaying normal component is
$$k_{z,m}=\begin{cases}
-\sqrt{D_m},&D_m>0\quad\text{(propagating)},\\
0,&D_m=0\quad\text{(grazing)},\\
-i\sqrt{-D_m},&D_m<0\quad\text{(evanescent)}.
\end{cases}$$
The last branch gives decay into $z<0$ with the stated time convention [4]. An evanescent component has no real 3D propagation vector.

For real orders, direction cosines are $(\alpha_m,\beta_m,\gamma_m)=\mathbf k_m/k$; thus $\alpha_i=a_i$ and $\beta_i=b_i$. The order candidates form a row $\alpha_m=\alpha_i+m\lambda/d$, $\beta_m=\beta_i$. This is the starting point for the later comparison with Harvey's diagram [2, 3].

The explorer starts with $\lambda=2$, $d=4$, $a_i=0.3$, $b_i=0.4$, and selected $m=+1$. At these settings that order propagates; the grazing and beyond presets increase $b_i$ to $0.6$ and $0.8$.


In [ ]:
# Find the shared wave_optics package in the project root.
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
# Relocate a cached package without changing the lesson's module reload order.
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

import importlib
import numpy as np
from IPython.display import display

# Rerunning setup disposes of this notebook's previous explorer.
if 'ewald_explorer' in globals():
    ewald_explorer.close()

import wave_optics.conical as conical
import wave_optics.conical_plotting as conical_plotting
import wave_optics.conical_view as conical_view
importlib.reload(conical)
importlib.reload(conical_plotting)
importlib.reload(conical_view)

ewald_explorer = conical_view.ConicalDiffractionExplorer()
ewald_explorer.set_example(0.4)


## 2. Ewald: describe the change in wavevector

What change connects the incoming vector to one outgoing vector? Define
$$\boxed{\Delta\mathbf k_m=\mathbf k_m-\mathbf k_i.}$$
This **scattering vector** is a difference, not the propagation vector of another light wave.

For real outgoing directions, substitute $\mathbf k_m=\Delta\mathbf k_m+\mathbf k_i$:
$$|\Delta\mathbf k_m+\mathbf k_i|^2=k^2.$$
In $\Delta\mathbf k$ coordinates this is a sphere of radius $k$, centered at $-\mathbf k_i$: the **Ewald sphere**. It passes through the origin because $|\mathbf k_i|=k$ [1].

The grating fixes only the tangential changes:
$$\boxed{\Delta k_{x,m}=mG,\qquad \Delta k_{y,m}=0.}$$
For each $m$, $\Delta k_z$ is free. Draw a line parallel to $\Delta k_z$ at $(mG,0)$: a **reciprocal-space rod**. Its sphere intersections give possible real outgoing vectors.

Select transmission into $z<0$ by
$$k_{z,m}=\Delta k_{z,m}+k_{z,i}<0,$$
with equality at grazing. The other real branch points into $z>0$ and is not used for the transmitted order. If a rod misses the sphere, the order has no real outgoing direction; its evanescent component remains defined by the normal component in section 1.

Writing $\Delta\mathbf k=mG\hat{\mathbf x}$ would incorrectly force $\Delta k_z=0$. Surface periodicity imposes no such condition. Pflüger and colleagues derive the grating-rod construction explicitly [1].

### Small calculation 1: construct the vector triangle

Use $m=+1$ at the conical preset. Compute the normalized center $C=-\mathbf k_i/k$ and endpoint $Q=\Delta\mathbf k_m/k$.

**Predict and rerun:** set $m=0$. Where does $Q$ move? Must a transmitted order have $\Delta k_z<0$?

In [ ]:
wavelength, d = 2.0, 4.0
a_i, b_i, m = 0.3, 0.4, 1
s_i = np.array([a_i, b_i, -np.sqrt(1-a_i**2-b_i**2)])
a_m = a_i+m*wavelength/d
r = 1-a_m**2-b_i**2

if r < -1e-12:
    print("This order has no real Ewald intersection. Its kz is imaginary.")
else:
    s_m = np.array([a_m, b_i, -np.sqrt(max(r, 0))])
    q_over_k = s_m-s_i
    C, O, Q = -s_i, np.zeros(3), q_over_k
    print("ki/k =", np.round(s_i, 6))
    print("km/k =", np.round(s_m, 6))
    print("Delta k/k  =", np.round(q_over_k, 6))
    print("C =", np.round(C, 6), "; O =", O, "; Q =", np.round(Q, 6))
    print("|Q-C| (sphere radius) =", np.linalg.norm(Q-C))
    print("Delta k_x/G =", q_over_k[0]/(wavelength/d), "; Delta k_y/k =", q_over_k[1])
    print("kz,m/k = Delta k_z/k + kz,i/k =", q_over_k[2]+s_i[2])

<details><summary>Answer</summary>

For $m=+1$,
$$\frac{\mathbf k_i}k=(0.3,0.4,-0.866025),\quad
\frac{\mathbf k_{+1}}k=(0.8,0.4,-0.447214),\quad
\frac{\Delta\mathbf k_{+1}}k=(0.5,0,0.418812).$$
Here $\Delta k_z>0$, yet $k_{z,m}<0$: this is transmission. A change and the resulting component need not have the same sign.

The triangle has $C\to O=\mathbf k_i/k$, $C\to Q=\mathbf k_m/k$, and $O\to Q=\Delta\mathbf k_m/k$. At $m=0$, $Q=O$ and the triangle collapses. The zero transmitted order continues the incident direction in our equal-medium model.

</details>

## 3. Follow the same order in the Ewald construction

The view uses $\Delta\mathbf k/k$, so the sphere has radius one and center $C=-\mathbf k_i/k$.

- **Left:** the full sphere, grating rods, and the selected vector triangle.
- **Right:** the rod plane $\Delta k_y=0$, whose intersection circle has radius $\rho=\sqrt{1-\beta_i^2}$.
- The solid lower arc selects transmission; the dashed remainder completes the circle.
- The orange rod stays selected when it has no real intersection; the real outgoing triangle disappears.

For conical incidence $C_y=-\beta_i\ne0$. The circle's center in the rod plane is **not the sphere center**: it is $(-\alpha_i,0,-\gamma_i)$. The cross in the right panel marks this section center.

**Try:** compare $m=+1$ and $m=0$ at the conical preset. Return to $m=+1$ and follow the grazing and beyond presets. Rotate the left view to see how the rod plane misses the sphere center.

In [ ]:
display(ewald_explorer.ewald_ui)

## 4. Connect the constructions

All three representations describe the same real outgoing orders:

| Representation | Order coordinate | Sphere center | Radius |
| --- | --- | --- | --- |
| Wavevector space | $\mathbf k_m$ | $\mathbf 0$ | $k$ |
| Direction-cosine space | $\hat{\mathbf k}_m=\mathbf k_m/k$ | $\mathbf 0$ | $1$ |
| Ewald scattering-vector space | $\Delta\mathbf k_m=\mathbf k_m-\mathbf k_i$ | $-\mathbf k_i$ | $k$ |

The exact relation is
$$\boxed{\hat{\mathbf k}_m=\frac{\Delta\mathbf k_m+\mathbf k_i}{k}.}$$

Translate Ewald coordinates by $\mathbf k_i$, divide by $k$, and project the outgoing hemisphere onto its tangential plane. You arrive at Harvey's disk. The Harvey view below shares this notebook's Ewald controls. Follow one selected order across these two views to give each operation a visible meaning.

For an evanescent candidate, the tangential coordinates and order label remain available, but neither sphere has a real outgoing point. The complex normal component describes its decay.

These constructions locate possible waves. Determining the coefficients $A_m$, their polarization, and their powers requires the grating's scattering response.

In [ ]:
display(ewald_explorer.harvey_ui)


## Final checkpoint

1. Why does a surface grating give rods rather than a full-vector constraint $\Delta\mathbf k=m\mathbf G$?
2. Can a transmitted order have $\Delta k_z>0$?
3. What happens to the Ewald triangle at $m=0$?
4. Where is the sphere center, and how do we recover Harvey's coordinates?

<details><summary>Answers</summary>

1. The grating fixes $\Delta k_x=mG$ and $\Delta k_y=0$, but leaves $\Delta k_z$ to the dispersion relation.
2. Yes: transmission is decided by $k_{z,m}=\Delta k_z+k_{z,i}<0$, not by the sign of $\Delta k_z$ alone.
3. In the common medium $\mathbf k_0=\mathbf k_i$, so $\Delta\mathbf k_0=0$ and the triangle collapses.
4. The sphere center is $-\mathbf k_i$ (or $-\mathbf k_i/k$ in the normalized plot). Translate by $\mathbf k_i$, divide by $k$, and project the outgoing hemisphere onto its tangential plane.

</details>


## References and conventions

1. **Pflüger et al. (2017)**, [*GISAXS on small periodic targets using large beams*](https://journals.iucr.org/m/issues/2017/04/00/hf5342/), §2 and Figure 3; [supplementary §2](https://journals.iucr.org/m/issues/2017/04/00/hf5342/hf5342sup1.pdf), Eqs. (5)–(13), for the rods. Exchange its periodicity and groove axes to match ours. The [2018 erratum](https://journals.iucr.org/m/issues/2018/04/00/hf5934/index.html) changes main-text Eq. (5) from arcsin to arctan; this notebook derives vectors directly. The [IUCr definition](https://dictionary.iucr.org/Ewald_sphere) uses the alternative reciprocal scale $1/\lambda$.
2. **Harvey and Vernold (1998)**, [*Description of Diffraction Grating Behavior in Direction Cosine Space*](https://doi.org/10.1364/AO.37.008158). [Publisher article page](https://opg.optica.org/ao/abstract.cfm?uri=ao-37-34-8158). Original direction-cosine treatment; Figures 1–3 connect the spatial geometry and its disk.
3. **Harvey and Pfisterer (2019)**, [*Understanding diffraction grating behavior: including conical diffraction and Rayleigh anomalies from transmission gratings*](https://doi.org/10.1117/1.OE.58.8.087105), §§2–3 and Figures 5–9. Their planar transmission equation uses the opposite order label with matching signed angles: $m_{\rm H}=-m$. In reflection observation diagrams the outward vector to the incident entry point is opposite to the incoming propagation vector.
4. **TU Delft**, [*Angular Spectrum Method*](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html#angular-spectrum-method), Eq. (375) and Figure 87: 3D dispersion and evanescence, with outgoing propagation toward $+z$.

The [reference map](Ewald_Construction_Reference_Map.md) records the checked source passages and convention changes. The earlier [Harvey and Ewald reference notebook](n_Wave_Equation_Gratings_Harvey_and_Ewald.ipynb) uses the opposite outgoing hemisphere.

To release this notebook's views, run <code>ewald_explorer.close()</code>. Rerunning setup closes the previous instance; rerun the later display cells to show its replacement. The saved notebook contains no executed outputs, images, or widget state.


## Selected freely accessible reading

Optional reading: start with the suggested passage. Freely accessible works are not necessarily in the public domain.

- **J. E. Harvey and R. N. Pfisterer. [Understanding diffraction grating behavior: including conical diffraction and Rayleigh anomalies from transmission gratings](https://www.spiedigitallibrary.org/journals/optical-engineering/volume-58/issue-08/087105/Understanding-diffraction-grating-behavior--including-conical-diffraction-and-Rayleigh/10.1117/1.OE.58.8.087105.full).** Optical Engineering 58, 087105 (2019); DOI: 10.1117/1.OE.58.8.087105. **Access:** Freely readable publisher full text; not public domain. **Read for:** Read Sections 2–4: signed directions and conical geometry.

- **D. Maystre. [Analytic Properties of Diffraction Gratings](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=33).** Chapter 2 in E. Popov (ed.), Gratings: Theory and Numeric Applications, second revisited edition (2014). **Access:** Free institutional book chapter; not public domain. **Read for:** Read Sections 2.3–2.5: order expansion and energy balance.

**Convention check:** Translate each source's order signs, propagation side, polarization assumptions and amplitude/power normalization to this notebook's stated model.

[Source catalogue and access notes](reading_sources.json).
